# Meridian MMM Platform — Step-by-Step Walkthrough

This notebook mirrors exactly what the Streamlit app does, step by step:

| App Page | Notebook Section |
|---|---|
| 📤 Data Upload | 1. Load & Validate Data |
| 🎓 Model Training | 2. Prepare Data → 3. Train Model |
| 📊 Results | 4. Diagnostics → 5. Results & Charts |
| 💰 Optimization | 6. Budget Optimization |

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Add the src directory to the Python path (same as the app does)
sys.path.insert(0, str(Path('../src').resolve()))

# Data loaders
from meridian_platform import MediaDataLoader, SalesDataLoader, PriorsConfigLoader

# Modeling & optimization (requires Meridian to be installed)
from meridian_platform.modeling.model_runner import MeridianModelRunner
from meridian_platform.optimization.budget_optimizer import BudgetOptimizer

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Setup complete')

---
## 1. Data Upload & Validation

The app's **Data Upload** page lets you upload CSVs for media, sales, and population data.  
Here we load the built-in sample files directly.

### 1a. Media Data

In [ ]:
SAMPLE_DIR = Path('../data/sample')

# Load using a DataFrame (same as the app does for uploaded files)
media_df = pd.read_csv(SAMPLE_DIR / 'sample_media_data.csv')
media_loader = MediaDataLoader(dataframe=media_df)
media_data = media_loader.load()

# Validate
validation = media_loader.validate(media_data)
print('Valid:', validation['valid'])
if validation['warnings']:
    for w in validation['warnings']:
        print(' ⚠️', w)

print(f"\nRows      : {len(media_data):,}")
print(f"Channels  : {media_data['touchpoint_name'].nunique()} — {media_data['touchpoint_name'].unique().tolist()}")
print(f"Geos      : {media_data['geo'].nunique()} — {media_data['geo'].unique().tolist()}")
print(f"Date range: {media_data['date'].min().date()} → {media_data['date'].max().date()}")

media_data.head()

### 1b. Sales Data

In [ ]:
sales_df = pd.read_csv(SAMPLE_DIR / 'sample_sales_data.csv')
sales_loader = SalesDataLoader(dataframe=sales_df)
sales_data = sales_loader.load()

validation = sales_loader.validate(sales_data)
print('Valid:', validation['valid'])

print(f"\nRows       : {len(sales_data):,}")
print(f"Geos       : {sales_data['geo'].nunique()}")
print(f"Total sales: ${sales_data['sales_value'].sum():,.0f}")
print(f"Avg sales  : ${sales_data['sales_value'].mean():,.0f}")

sales_data.head()

### 1c. Population Data

In [ ]:
population_df = pd.read_csv(SAMPLE_DIR / 'sample_population_data.csv')
print(population_df)

### 1d. Visualise the raw data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Spend by channel ---
spend_by_channel = media_data.groupby('touchpoint_name')['spend'].sum().sort_values()
spend_by_channel.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Total Spend by Channel')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].set_xlabel('Spend')

# --- Metric (impressions/GRPs) by channel ---
metric_by_channel = media_data.groupby('touchpoint_name')['metric_value'].sum().sort_values()
metric_by_channel.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Total Metric (Impressions/GRPs) by Channel')
axes[1].set_xlabel('Metric Value')

# --- Sales over time (all geos combined) ---
sales_over_time = sales_data.groupby('date')['sales_value'].sum()
sales_over_time.plot(ax=axes[2], color='darkgreen', linewidth=2)
axes[2].set_title('Total Sales Over Time')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.suptitle('Raw Data Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Model Training — Prepare Data

The app's **Model Training** page first aggregates your data to weekly and converts it into the Meridian `InputData` format.

In [ ]:
# Initialise the runner (same call as the app makes)
runner = MeridianModelRunner(
    media_data=media_data,
    sales_data=sales_data,
    population_data=population_df,
    priors_config=None   # no custom priors
)

# Prepare data — aggregates daily → weekly, then builds Meridian InputData
input_data = runner.prepare_data(aggregate_weekly=True)

print(f"KPI shape   : {input_data.kpi.shape}   (geos × time-periods)")
print(f"Media shape : {input_data.media.media_value.shape}  (geos × time × channels)")
print(f"Channels    : {input_data.media.media_channel.values.tolist()}")

### Inspect the wide-format pivot used internally
The runner pivots the media data so each channel becomes its own column — this is what Meridian ingests.

In [ ]:
# Reconstruct the weekly data for inspection
weekly_media = runner.media_data.copy()   # already aggregated to weekly by prepare_data()
weekly_spend_total = weekly_media.groupby('touchpoint_name')['spend'].sum().sort_values(ascending=False)

print('Weekly spend totals per channel:')
for ch, sp in weekly_spend_total.items():
    print(f'  {ch:<25} ${sp:>12,.0f}')

---
## 3. Model Training — Train

The app runs MCMC sampling via Meridian's `sample_prior()` and `sample_posterior()`.  
Adjust `n_warmup`, `n_samples`, and `n_chains` to trade speed vs accuracy.

> **Note:** This cell can take several minutes. Reduce `n_warmup`/`n_samples` for a quick test run.

In [ ]:
# Configuration mirrors the app's default slider values
N_WARMUP  = 500   # app default: 500
N_SAMPLES = 500   # app default: 500
N_CHAINS  = 2     # app default: 2
MAX_LAG   = 8     # app default: 8 (weeks of adstock)
SEED      = 42

model_spec = runner.create_model_spec()

trained_model = runner.train(
    n_warmup=N_WARMUP,
    n_samples=N_SAMPLES,
    n_chains=N_CHAINS,
    seed=SEED,
)
print('✅ Training complete')

---
## 4. Model Diagnostics

The app shows R-hat convergence metrics.  
R-hat < 1.2 means the MCMC chains converged.

In [ ]:
diagnostics = runner.get_diagnostics()

print('Convergence summary')
print(f"  R-hat (max) : {diagnostics['summary']['r_hat_max']:.4f}  (< 1.2 is good)")
print(f"  R-hat (mean): {diagnostics['summary']['r_hat_mean']:.4f}")
print(f"  Converged   : {'✅ Yes' if diagnostics['convergence']['overall_converged'] else '❌ No'}")

if 'rhat_details' in diagnostics:
    display(diagnostics['rhat_details'])

---
## 5. Model Results

This corresponds to the **📊 Results** page — ROI by channel.

In [ ]:
results = runner.get_results()

roi_df = (
    pd.DataFrame(
        [{'Channel': ch, 'ROI': roi}
         for ch, roi in results['roi_adjusted'].items()]
    )
    .sort_values('ROI', ascending=False)
    .reset_index(drop=True)
)

print('ROI by channel (mean posterior estimate):')
display(roi_df.style.format({'ROI': '{:.2f}x'}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['green' if roi >= 1 else 'tomato' for roi in roi_df['ROI']]
roi_df.sort_values('ROI').plot(
    kind='barh', x='Channel', y='ROI',
    ax=ax, legend=False, color=colors
)
ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5, label='Break-even (1x)')
ax.set_title('Return on Investment by Channel', fontsize=14, fontweight='bold')
ax.set_xlabel('ROI  ($ sales per $ spent)')
ax.legend()
plt.tight_layout()
plt.show()

### Meridian visualizer charts (requires `meridian` package)
The app uses Meridian's built-in Altair charts. You can render them here too:

In [ ]:
try:
    from meridian.analysis.visualizer import ModelDiagnostics, ModelFit, MediaEffects, MediaSummary

    summary_viz = MediaSummary(trained_model)
    summary_viz.plot_roi_bar_chart(include_ci=True)   # returns an Altair chart
except ImportError:
    print('Meridian visualizer not available — skipping built-in charts')

---
## 6. Budget Optimization

This corresponds to the **💰 Optimization** page.

In [ ]:
optimizer = BudgetOptimizer(trained_model=trained_model)

TOTAL_BUDGET = 1_000_000   # $1 M — matches the app's default
CONSTRAINTS  = 'medium'    # ±50% from current spend per channel

result = optimizer.optimize_roi(TOTAL_BUDGET, CONSTRAINTS)

alloc_df = (
    pd.DataFrame(
        [{'Channel': ch, 'Budget': budget}
         for ch, budget in result['allocation'].items()]
    )
    .sort_values('Budget', ascending=False)
    .reset_index(drop=True)
)

print(f"Optimized allocation  (total budget ${TOTAL_BUDGET:,.0f}, constraints: {CONSTRAINTS})")
display(alloc_df.style.format({'Budget': '${:,.0f}'}))

m = result['metrics']
print(f"\nProjected total sales : ${m['total_sales']:,.0f}")
print(f"Projected total spend : ${m['total_spend']:,.0f}")
print(f"Projected overall ROI : {m['total_roi']:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

alloc_df.sort_values('Budget').plot(
    kind='barh', x='Channel', y='Budget',
    ax=ax, legend=False, color='mediumseagreen'
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_title(f'Optimised Budget Allocation  (total ${TOTAL_BUDGET/1e6:.1f}M)', fontsize=14, fontweight='bold')
ax.set_xlabel('Recommended Budget')
plt.tight_layout()
plt.show()

---
## Summary

| Step | What happened |
|---|---|
| Load data | `MediaDataLoader`, `SalesDataLoader` parse & validate CSVs |
| Prepare data | Weekly aggregation → Meridian `InputData` (geos × time × channels) |
| Train model | `sample_prior()` + `sample_posterior()` MCMC via Meridian |
| Diagnostics | R-hat convergence check (<1.2 is healthy) |
| Results | ROI per channel extracted from posterior samples |
| Optimise | `BudgetOptimizer.optimize_roi()` finds the spend mix that maximises ROI |

Every cell maps directly to a section of the Streamlit app — the notebook is a fully executable version of it.